# F02-P3 Threat

**Ecosystem Threat: Observed Disturbance, Ecosystem Pressure and Hazards, and Business-as-Usual Projection**

This section identifies current and potential threats within the project area, including human-induced disturbance, natural disturbance, and climate-related hazards. F02-P3 Threat addresses two questions: **“What are the potential threats?”** and **“What may happen if these threats are not mitigated?”**

> **Not runnable yet.** The analysis logic below is implemented in Python, but file access is currently stubbed.

**Status.** Complete as scoped for the **All Ecosystems** section.

**Known limitation.** Areas exposed to high disaster risk may affect the feasibility and effectiveness of NbS interventions. High-risk areas should therefore be carefully considered when selecting intervention locations.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass, replace
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
import pandas as pd

from pyproj import Geod
from rasterio.mask import raster_geometry_mask
from rasterio.mask import mask
from rasterio.warp import reproject, Resampling
from scipy.ndimage import distance_transform_edt
from config import *
from common import *

---
# Observed Ecosystem Disturbance


---
## 7.1 Historical Deforestation (2014 to 2024)

Reports how much forest the AOI lost between 2014 and 2024 and the annual rate using Puyravaud.

**Data.** `FC2014.tif` (binary forest cover 2014, Tier 1-2) and `LC2024.tif` (20 class land cover
2024). Both dates use the same Tier 1-2 forest definition, through `forest_mask_2014` and
`forest_mask_2024` in `common.py`.

**Decisions locked.**

- `A2 = A1 - loss_ha`, gross loss of 2014 forest. Forest gain on non forest 2014 land is
  excluded, so the rate stays consistent with the reported loss.
- No comparison against a national rate. The component reports the site on its own terms. The
  national lookup, its CSV and the similar band threshold have been removed from the tool.
- The rate is shown to one decimal. Screening precision, not a monitoring figure.

**Puyravaud (2003).** `rate = (1 / (t2 - t1)) * ln(A2 / A1) * 100`

**Two edge cases decided here, not in the spec.**

1. No loss at all. The Puyravaud formula returns 0, which is correct but reads oddly as "an
   average of 0.0% per year", so the component uses a dedicated sentence instead.
2. Total loss, `A2 == 0`. All 2014 forest is gone and `ln(0)` is undefined. The component
   reports `rate_pct = None` and says the area lost all of its forest, rather than emitting an
   infinite rate.

**Example render.**

> Between 2014 and 2024, the project area lost 180 ha of forest, an average of 1.6% per year.

**Downstream use.** Observed loss is the empirical basis of the trajectory (Axis 2) and a direct
threat intensity signal. The AUD pathway signal now comes from the modelled risk in 1.6 rather
than from a national rate comparison here.

In [ ]:
def analyze_historical_deforestation(aoi: AOI) -> ComponentResult:
    """Component 1.5. Forest loss 2014 to 2024 and its annual rate."""
    f2014 = forest_mask_2014(aoi)
    f2024 = forest_mask_2024(aoi)

    if f2014.is_empty:
        return not_applicable(
            "1.5 Historical Deforestation",
            "No forest was present in this project area in 2014, so a deforestation rate "
            "cannot be calculated.",
        )

    # Derived output rasters: forest at each date and the gross-loss grid.
    _loss = f2014.mask & ~f2024.mask
    loss_slice = RasterSlice(
        np.ma.masked_array(_loss.astype("float32"), mask=False),
        f2014.pixel_area_ha, f2014.transform, f2014.crs,
    )
    forest_rasters = {
        "1.5_forest_2014": f2014.as_slice(),
        "1.5_forest_2024": f2024.as_slice(),
        "1.5_forest_loss": loss_slice,
    }

    a1 = f2014.area_ha
    # Gross loss of 2014 forest. Gain elsewhere is deliberately not netted off.
    loss_ha = int((f2014.mask & ~f2024.mask).sum()) * f2014.pixel_area_ha
    a2 = a1 - loss_ha

    if loss_ha <= 0:
        rate_pct = 0.0
        narrative = "No forest loss was detected in this project area between 2014 and 2024."
    elif a2 <= 0:
        rate_pct = None  # ln(0) guard, see the note above
        narrative = (
            f"Between 2014 and 2024, the project area lost all of its {fmt_ha(a1)} of forest."
        )
    else:
        rate_pct = abs((1.0 / DEFOR_PERIOD_YEARS) * math.log(a2 / a1) * 100.0)
        narrative = (
            f"Between 2014 and 2024, the project area lost {fmt_ha(loss_ha)} of forest, an "
            f"average of {rate_pct:.1f}% per year."
        )

    return ComponentResult(
        component="1.5 Historical Deforestation",
        applicable=True,
        narrative=narrative,
        tables={},
        values={
            "forest_2014_ha": a1,
            "forest_2024_ha": a2,
            "loss_ha": loss_ha,
            "rate_pct": rate_pct,
        },
        rasters=forest_rasters,
    )


results["1.5"] = analyze_historical_deforestation(aoi)
show_result(results["1.5"])

---
## 7.2 Historical Burned Area

Reports how much of the AOI has burned over the GABAM record, 2014 to 2024, as an annual series
(bar chart) and one burned footprint.

**Data.** GABAM annual burned maps (`GABAM_RASTER_TEMPLATE`), one binary raster per year, value
1 = burned and 0 = nodata, about 30 m, EPSG:4326. Eleven years, 2014 to 2024.

**Two totals, on purpose.** A pixel can burn in more than one year, so:

- The headline is the **union**: the area that burned at least once. It is always at most the AOI
  area and reads as the true burned footprint.
- The bar chart is **per year**: a pixel that reburns is counted in each year it burns, which is
  what an annual series should show.
- The **sum** of the annual areas is also kept in `values` (`burned_sum_ha`). It can exceed the
  AOI area and measures burn events, not footprint, so it must not be read as an area.

**Decisions locked.**

- Burned = the valid (non nodata) pixels, since GABAM encodes 0 as nodata.
- All years are aligned to the first year's grid (`like=`), so the union is an exact pixel-wise OR.
- Nearest resampling only, never interpolate a burned or not-burned flag.

**Example render.**

> Between 2014 and 2024, 1,240 ha of the project area burned at least once, about 8.6% of the area.

**Downstream use.** A fire history raises permanence risk and points to fire-driven degradation;
it also supports fire-management activities in the pathway logic.

In [ ]:
records = []

for year in YEARS:

    path = GABAM_FOLDER / f"GABAM_{year}.tif"

    with rasterio.open(path) as src:

        geom = aoi.to_crs(src.crs).geometry

        data, transform = mask(
            src,
            geom,
            crop=True,
            filled=False
        )

        burned = (
            ~np.ma.getmaskarray(data[0])
            & (data[0].data == BURNED_VALUE)
        )

        # EPSG:4326 / geographic raster
        if src.crs.is_geographic:

            burned_ha = 0

            for row in range(burned.shape[0]):

                north = transform.f + row * transform.e
                south = north + transform.e

                area, _ = GEOD.polygon_area_perimeter(
                    [
                        transform.c,
                        transform.c + transform.a,
                        transform.c + transform.a,
                        transform.c
                    ],
                    [north, north, south, south]
                )

                burned_ha += (
                    burned[row].sum()
                    * abs(area)
                    / 10000
                )

        else:

            pixel_ha = (
                abs(transform.a * transform.e)
                / 10000
            )

            burned_ha = (
                burned.sum()
                * pixel_ha
            )

    records.append({
        "year": year,
        "burned_area_ha": burned_ha,
        "burned_pct_aoi": burned_ha / aoi_area_ha * 100
    })


# =============================================================================
# SUMMARY
# =============================================================================

annual = pd.DataFrame(records)

cumulative_ha = annual["burned_area_ha"].sum()
average_ha = annual["burned_area_ha"].mean()

peak = annual.loc[
    annual["burned_area_ha"].idxmax()
]


# =============================================================================
# NARRATIVE
# =============================================================================

if cumulative_ha > 0:

    narrative = (
        f"The project area recorded a cumulative burned area of "
        f"{cumulative_ha:,.2f} ha between 2014 and 2024, "
        f"with an average annual burned area of {average_ha:,.2f} ha. "
        f"The highest annual burned area was recorded in "
        f"{int(peak['year'])}, when {peak['burned_area_ha']:,.2f} ha "
        f"was affected by fire."
    )

else:

    narrative = (
        "No burned area was detected within the project area "
        "between 2014 and 2024."
    )


# =============================================================================
# OUTPUT
# =============================================================================

annual["burned_area_ha"] = annual["burned_area_ha"].round(2)
annual["burned_pct_aoi"] = annual["burned_pct_aoi"].round(2)

print(annual.to_string(index=False))
print()
print(narrative)

## 7.3 Land Degradation

Land degradation is assessed following the **UNCCD SDG 15.3.1 framework**, which integrates three sub-indicators: **land-cover change, land productivity, and soil organic carbon (SOC)**. Regional and globally available spatial datasets are used to provide a harmonized assessment across Southeast Asia.

- **Land cover:** Land-cover change is assessed between **2014 and 2024** using the Regional Land Cover Monitoring System. Forest and non-forest transitions are reclassified as **Degraded, Stable, or Improved** according to the direction of land-cover change.
- **Land productivity:** Productivity is assessed using an annual NDVI-based dataset derived from fused Landsat-8 and MODIS observations for **2013–2022**. Multi-Temporal Image Differencing is used to estimate productivity status, while the **Mann-Kendall test** is used to evaluate the significance of temporal trends.
- **Soil organic carbon:** SOC is assessed using a **30 m OpenLandMap dataset**. Relative changes in SOC stocks between assessment periods are used to classify each area as **Degraded, Stable, or Improved**, representing changes in soil and ecosystem condition.
- **Final degradation status:** The three sub-indicators are integrated using the UNCCD **One-Out-All-Out (1OAO)** approach. If any sub-indicator indicates degradation, the overall status is classified as degraded. Degradation occurring inside forest areas may be reported as **forest degradation**, while degradation outside forest areas may be reported as **land degradation**.

Because the three indicators represent different observation periods, the assessment also identifies **Degraded – Potential False Positive** and **Degraded – Potential False Negative** classes. These flags highlight cases where temporal differences among land cover, SOC, and land-productivity observations may produce an uncertain degradation classification and where field observations or other ground-truth information are recommended for interpretation.

In [ ]:
# ---------- READ AOI ----------
aoi = gpd.read_file(aoi_path)
aoi_geom = [geom for geom in aoi.geometry]

# ---------- STEP 1: AOI clips forest cover and land degradation ----------
with rasterio.open(forest_path) as src:
    forest_arr, forest_transform = mask(src, aoi_geom, crop=True, nodata=0, filled=True)
    forest_profile = src.profile.copy()
    forest_crs = src.crs

forest_arr = forest_arr[0].astype("float32")

forest_profile.update(
    height=forest_arr.shape[0],
    width=forest_arr.shape[1],
    transform=forest_transform,
    count=1,
    dtype="float32",
    nodata=0,
    compress="lzw",
)

with rasterio.open(forest_out, "w", **forest_profile) as dst:
    dst.write(forest_arr, 1)

with rasterio.open(degradation_path) as src:
    deg_arr, deg_transform = mask(src, aoi_geom, crop=True, nodata=0, filled=True)
    deg_crs = src.crs

deg_arr = deg_arr[0].astype("float32")

deg_profile = forest_profile.copy()
deg_profile.update(
    height=deg_arr.shape[0],
    width=deg_arr.shape[1],
    transform=deg_transform,
)

with rasterio.open(degradation_aoi_out, "w", **deg_profile) as dst:
    dst.write(deg_arr, 1)

print("Step 1 done: forest shape =", forest_arr.shape, "| degradation shape =", deg_arr.shape)

# Align clipped degradation to the clipped forest grid (same extent, cell size, shape)
deg_aligned = np.zeros(forest_arr.shape, dtype="float32")
reproject(
    source=deg_arr,
    destination=deg_aligned,
    src_transform=deg_transform,
    src_crs=deg_crs,
    src_nodata=0,
    dst_transform=forest_transform,
    dst_crs=forest_crs,
    dst_nodata=0,
    resampling=Resampling.nearest,
)

# ---------- STEP 2: Forest pixel 1 clips land degradation -> degradation inside forest ----------
forest_1 = forest_arr == 1

deg_inside = np.where(forest_1 & (deg_aligned > 0), deg_aligned, 0).astype("float32")

with rasterio.open(inside_out, "w", **forest_profile) as dst:
    dst.write(deg_inside, 1)

print("Step 2 done: degradation pixels inside forest =", int((deg_inside > 0).sum()))

# ---------- STEP 3: Mask outside forest pixel 1 -> degradation outside forest ----------
deg_outside = np.where(~forest_1 & (deg_aligned > 0), deg_aligned, 0).astype("float32")

with rasterio.open(outside_out, "w", **forest_profile) as dst:
    dst.write(deg_outside, 1)

print("Step 3 done: degradation pixels outside forest =", int((deg_outside > 0).sum()))

# ---------- STEP 4: Total area inside forest and outside forest (class 1 to 5) ----------
pixel_area_m2 = abs(forest_transform.a * forest_transform.e)  # EPSG:54034 is equal-area

rows = []
for value in range(1, 6):
    inside_count = int((deg_inside == value).sum())
    outside_count = int((deg_outside == value).sum())
    rows.append({
        "class": value,
        "inside_forest_pixels": inside_count,
        "inside_forest_ha": round(inside_count * pixel_area_m2 / 10000, 2),
        "outside_forest_pixels": outside_count,
        "outside_forest_ha": round(outside_count * pixel_area_m2 / 10000, 2),
    })

df = pd.DataFrame(rows)

total_row = pd.DataFrame([{
    "class": "Total",
    "inside_forest_pixels": df["inside_forest_pixels"].sum(),
    "inside_forest_ha": round(df["inside_forest_ha"].sum(), 2),
    "outside_forest_pixels": df["outside_forest_pixels"].sum(),
    "outside_forest_ha": round(df["outside_forest_ha"].sum(), 2),
}])
df = pd.concat([df, total_row], ignore_index=True)

df.to_csv(csv_out, index=False)
print(df)
# =============================================================================
# STEP 5: NARRATIVE
# =============================================================================

pixel_area_ha = pixel_area_m2 / 10000


# -----------------------------------------------------------------------------
# Whole AOI land degradation classes
# -----------------------------------------------------------------------------

degraded_ha = (
    (deg_aligned == 1).sum()
    * pixel_area_ha
)

stable_ha = (
    (deg_aligned == 2).sum()
    * pixel_area_ha
)

improved_ha = (
    (deg_aligned == 3).sum()
    * pixel_area_ha
)

false_positive_ha = (
    (deg_aligned == 4).sum()
    * pixel_area_ha
)

false_negative_ha = (
    (deg_aligned == 5).sum()
    * pixel_area_ha
)


# -----------------------------------------------------------------------------
# Total degradation
#
# 1 = Degraded
# 4 = Degraded - potential false positive
# 5 = Degraded - potential false negative
# -----------------------------------------------------------------------------

total_degradation_ha = (
    degraded_ha
    + false_positive_ha
    + false_negative_ha
)


# -----------------------------------------------------------------------------
# Degradation inside forest
#
# deg_masked already contains only pixels where forest == 1
# -----------------------------------------------------------------------------

degraded_inside_forest_ha = (
    np.isin(
        deg_masked,
        [1, 4, 5]
    ).sum()
    * pixel_area_ha
)


# -----------------------------------------------------------------------------
# Degradation outside forest
# -----------------------------------------------------------------------------

degraded_outside_forest_ha = (
    total_degradation_ha
    - degraded_inside_forest_ha
)


# -----------------------------------------------------------------------------
# Narrative
# -----------------------------------------------------------------------------

narrative = (
    f"A total of {total_degradation_ha:,.2f} ha within the project area "
    f"showed signs of land degradation between 2014 and 2024. "
    f"Of this area, {false_positive_ha:,.2f} ha was flagged as "
    f"Potential False Positive and {false_negative_ha:,.2f} ha was "
    f"flagged as Potential False Negative, respectively."
    "\n\n"
    f"Of the total degradation-related area, "
    f"{degraded_inside_forest_ha:,.2f} ha occurred within forest areas, "
    f"while {degraded_outside_forest_ha:,.2f} ha occurred outside forest areas."
)


# -----------------------------------------------------------------------------
# Improved condition
# -----------------------------------------------------------------------------

if improved_ha > 0:

    narrative += (
        "\n\n"
        f"In contrast, {improved_ha:,.2f} ha within the project area "
        f"showed positive changes in land condition between 2014 and 2024."
    )


# -----------------------------------------------------------------------------
# Stable condition
# -----------------------------------------------------------------------------

if stable_ha > 0:

    narrative += (
        "\n\n"
        f"The remaining {stable_ha:,.2f} ha was classified as stable, "
        f"indicating no significant improvement or deterioration in the "
        f"assessed land condition indicators between 2014 and 2024."
    )


# -----------------------------------------------------------------------------
# Display
# -----------------------------------------------------------------------------

print()
print("=" * 70)
print("LAND DEGRADATION SUMMARY")
print("=" * 70)

print(
    f"Total degradation          : "
    f"{total_degradation_ha:,.2f} ha"
)

print(
    f"Potential false positive   : "
    f"{false_positive_ha:,.2f} ha"
)

print(
    f"Potential false negative   : "
    f"{false_negative_ha:,.2f} ha"
)

print(
    f"Degraded inside forest     : "
    f"{degraded_inside_forest_ha:,.2f} ha"
)

print(
    f"Degraded outside forest    : "
    f"{degraded_outside_forest_ha:,.2f} ha"
)

print(
    f"Improved                   : "
    f"{improved_ha:,.2f} ha"
)

print(
    f"Stable                     : "
    f"{stable_ha:,.2f} ha"
)

print()
print(narrative)

---
# Ecosystem Pressure and Hazards

## 7.4 Peatland Ecosystem

Reports the drivers of peatland disturbance.

**Data.** `forest_drivers_v3.tif: pixel ranging from 1 to 11 ` This data comes from
Bart Slagter, et al 2026 https://doi.org/10.21203/rs.3.rs-7424252/v1
Which only focuses on classifying the key drivers of forest disturbances and 
may not include all potential causes of deforestation.


**Calibration warning.**  
The drainage canal dataset developed by Dadap et al. (2021) captures drainage canals located across both peatland and non-peatland areas. Therefore, the dataset was masked using `ecosystem_v3.tif` to retain only drainage canals occurring within mapped peatland areas in Southeast Asia.

Drainage pressure, however, refers to the proximity of drainage canals to the project area, including canals located outside the project boundary that may still influence peatland conditions within the project area. This follows previous findings indicating that:

a) Astiani et al. (2017) found that canals can influence water-table depth up to 500 m from the surrounding area.

b) Wedeux et al. (2020) found that canal networks may affect forest biomass growth up to 1 km away.

## Peatland Disturbance

| Indicator | Value |
|---|---:|
| **Total area** | **24,694.84 ha** |
| **Remaining peatland forest** | **1,182.02 ha (4.79%)** |
| **Disturbed** | **165.43 ha (0.67%)** |
| **Converted / Loss** | **4,968.08 ha (20.12%)** |

### Peatland Disturbance Drivers

| Driver | Status |
|---|---:|
| **Canal proximity** | **High** |
| **Nearest canal** | **0.00 m** |
| **Drainage pressure** | **High** |
| **Fire risk** | **High** |

In [ ]:
def read_raster(
    path,
    reference=None,
    reference_transform=None,
    reference_crs=None
):

    with rasterio.open(path) as src:

        polygon = (
            aoi.to_crs(src.crs)
            if aoi.crs != src.crs
            else aoi
        )

        try:

            data, raster_transform = mask(
                src,
                polygon.geometry,
                crop=True,
                filled=False
            )

            return (
                data[0],
                raster_transform,
                src.crs
            )

        except ValueError as e:

            if (
                "Input shapes do not overlap raster" not in str(e)
                or reference is None
            ):
                raise

            # No raster coverage inside AOI → return zero
            data = np.ma.array(
                np.zeros(
                    reference.shape,
                    dtype="float32"
                ),
                mask=False
            )

            return (
                data,
                reference_transform,
                reference_crs
            )

# =============================================================================
# AREA CALCULATION
# EPSG:4326 -> hectares
# =============================================================================

GEOD = Geod(
    ellps="WGS84"
)


def area_ha(
    binary_mask,
    transform
):

    binary_mask = np.asarray(
        binary_mask,
        dtype=bool
    )

    total_m2 = 0.0

    pixel_width = abs(
        transform.a
    )

    for row in range(
        binary_mask.shape[0]
    ):

        count = np.count_nonzero(
            binary_mask[row]
        )

        if count == 0:
            continue

        north = (
            transform.f
            + row * transform.e
        )

        south = (
            north
            + transform.e
        )

        west = transform.c
        east = west + pixel_width

        pixel_m2, _ = (
            GEOD.polygon_area_perimeter(
                [west, east, east, west],
                [north, north, south, south]
            )
        )

        total_m2 += (
            count
            * abs(pixel_m2)
        )

    return (
        total_m2 / 10000
    )


# =============================================================================
# LOAD RASTERS
# =============================================================================

ecosystem, transform, crs = read_raster(
    ECOSYSTEM
)

historical, historical_transform, _ = read_raster(
    HISTORICAL,
    ecosystem,
    transform,
    crs
)

forest2024, forest2024_transform, _ = read_raster(
    FOREST_2024,
    ecosystem,
    transform,
    crs
)

disturbance, disturbance_transform, _ = read_raster(
    DISTURBANCE,
    ecosystem,
    transform,
    crs
)

canal_density, canal_transform, _ = read_raster(
    PEAT_CANALS_DENSITY,
    ecosystem,
    transform,
    crs
)

fire_risk, fire_transform, _ = read_raster(
    FIRE_RISK,
    ecosystem,
    transform,
    crs
)

drainage_canals, drainage_transform, _ = read_raster(
    DRAINAGE_CANALS,
    ecosystem,
    transform,
    crs
)

# =============================================================================
# 1. TOTAL PEATLAND
#
# AOI
# -> ecosystem_v3 == 3
# =============================================================================

peatland_mask = (
    ecosystem == PEATLAND
)

total_area = area_ha(
    peatland_mask,
    transform
)


# =============================================================================
# 2. REMAINING PEATLAND FOREST
#
# Peatland
# -> historical_deforestation_v3 == 1
# =============================================================================

remaining_mask = (
    peatland_mask
    & (
        historical
        == REMAINING_FOREST
    )
)

remaining_area = area_ha(
    remaining_mask,
    transform
)


# =============================================================================
# 3. CURRENT PEATLAND FOREST
#
# Peatland
# -> forest_2024_v3 == 1
# =============================================================================

current_peatland_mask = (
    peatland_mask
    & (
        forest2024
        == CURRENT_FOREST
    )
)


# =============================================================================
# 4. DISTURBED PEATLAND
#
# Current peatland forest
# -> forest_disturbance_v3 > 0
# =============================================================================

disturbed_mask = (
    current_peatland_mask
    & (
        disturbance
        > DISTURBANCE_THRESHOLD
    )
)

disturbed_area = area_ha(
    disturbed_mask,
    transform
)


# =============================================================================
# 5. CONVERTED / LOSS
#
# Peatland
# -> historical_deforestation_v3 == 2
# =============================================================================

loss_mask = (
    peatland_mask
    & (
        historical
        == FOREST_LOSS
    )
)

loss_area = area_ha(
    loss_mask,
    transform
)


# =============================================================================
# PERCENTAGES
# =============================================================================

def percentage(area):

    if total_area == 0:
        return 0

    return (
        area
        / total_area
        * 100
    )


remaining_percentage = percentage(
    remaining_area
)

disturbed_percentage = percentage(
    disturbed_area
)

loss_percentage = percentage(
    loss_area
)


# =============================================================================
# CANAL PROXIMITY
#
# drainage_canal_v3.tif
# pixel 1 = drainage canal
#
# Current peatland forest -> nearest canal
#
# <= 500 m       = High
# > 500-1000 m   = Medium
# > 1000 m       = Low
# =============================================================================

from rasterio.features import shapes
from shapely.geometry import shape


def mask_to_geometry(
    binary_mask,
    transform,
    crs
):

    geometries = [
        shape(geom)
        for geom, value in shapes(
            binary_mask.astype(
                np.uint8
            ),
            mask=binary_mask,
            transform=transform
        )
        if value == 1
    ]

    if not geometries:
        return None

    return (
        gpd.GeoSeries(
            geometries,
            crs=crs
        )
        .union_all()
    )


def canal_proximity_level():

    # Current peatland forest
    if not np.any(
        current_peatland_mask
    ):
        return "Not identified", None


    # Drainage canal pixel = 1
    canal_mask = (
        drainage_canals == 1
    )


    if not np.any(
        canal_mask
    ):
        return "Low", None


    # Convert raster masks to geometry
    peat_forest_geometry = mask_to_geometry(
        current_peatland_mask,
        transform,
        raster_crs
    )

    canal_geometry = mask_to_geometry(
        canal_mask,
        drainage_transform,
        drainage_crs
    )


    if (
        peat_forest_geometry is None
        or canal_geometry is None
    ):
        return "Low", None


    # Use projected CRS for metre distance
    metric_crs = (
        aoi.estimate_utm_crs()
    )


    peat_metric = (
        gpd.GeoSeries(
            [peat_forest_geometry],
            crs=raster_crs
        )
        .to_crs(metric_crs)
        .iloc[0]
    )


    canal_metric = (
        gpd.GeoSeries(
            [canal_geometry],
            crs=drainage_crs
        )
        .to_crs(metric_crs)
        .iloc[0]
    )


    # Nearest distance
    distance_m = (
        peat_metric.distance(
            canal_metric
        )
    )


    if distance_m <= 500:

        level = "High"

    elif distance_m <= 1000:

        level = "Medium"

    else:

        level = "Low"


    return (
        level,
        float(distance_m)
    )


canal_proximity, canal_distance_m = (
    canal_proximity_level()
)
# =============================================================================
# 7. DRAINAGE PRESSURE
#
# Peatland
# -> peat_canals_density_v3
#
# 3 = High
# 2 = Medium
# 1 = Low
#
# UI uses highest class present.
# =============================================================================

drainage_values = np.unique(
    canal_density[
        peatland_mask
    ]
)


if 3 in drainage_values:

    drainage_pressure = (
        "High"
    )

elif 2 in drainage_values:

    drainage_pressure = (
        "Medium"
    )

elif 1 in drainage_values:

    drainage_pressure = (
        "Low"
    )

else:

    drainage_pressure = (
        "Not identified"
    )


# =============================================================================
# 8. FIRE RISK
#
# Peatland
# -> fire_risk_v3
#
# 4 = High
# 3 = Medium
# 2 = Low
# 1 = Very low
# 0 / NoData = No risk
#
# UI uses highest class present.
# =============================================================================

fire_values = np.unique(
    fire_risk[
        peatland_mask
    ]
)


if 4 in fire_values:

    fire_risk_level = (
        "High"
    )

elif 3 in fire_values:

    fire_risk_level = (
        "Medium"
    )

elif 2 in fire_values:

    fire_risk_level = (
        "Low"
    )

elif 1 in fire_values:

    fire_risk_level = (
        "Very low"
    )

else:

    fire_risk_level = (
        "No risk"
    )


# =============================================================================
# BACKEND RESULT
# =============================================================================

result = {

    "peatland": {

        "total_area_ha":
            round(
                total_area,
                2
            ),

        "remaining_forest": {

            "area_ha":
                round(
                    remaining_area,
                    2
                ),

            "percentage":
                round(
                    remaining_percentage,
                    2
                )
        },

        "disturbed": {

            "area_ha":
                round(
                    disturbed_area,
                    2
                ),

            "percentage":
                round(
                    disturbed_percentage,
                    2
                )
        },

        "converted_loss": {

            "area_ha":
                round(
                    loss_area,
                    2
                ),

            "percentage":
                round(
                    loss_percentage,
                    2
                )
        },

        "drivers": {

            "canal_proximity":
                canal_proximity,

            "canal_distance_m":
                (
                    round(
                        canal_distance_m,
                        2
                    )
                    if canal_distance_m
                    is not None
                    else None
                ),

            "drainage_pressure":
                drainage_pressure,

            "fire_risk":
                fire_risk_level
        }
    }
}


# =============================================================================
# FINAL OUTPUT
# =============================================================================

r = result[
    "peatland"
]


print()
print("=" * 75)
print("PEATLAND DISTURBANCE")
print("=" * 75)


print(
    f"Total area                : "
    f"{r['total_area_ha']:,.2f} ha"
)


print(
    f"Remaining peatland forest : "
    f"{r['remaining_forest']['area_ha']:,.2f} ha "
    f"({r['remaining_forest']['percentage']:.2f}%)"
)


print(
    f"Disturbed                 : "
    f"{r['disturbed']['area_ha']:,.2f} ha "
    f"({r['disturbed']['percentage']:.2f}%)"
)


print(
    f"Converted / Loss          : "
    f"{r['converted_loss']['area_ha']:,.2f} ha "
    f"({r['converted_loss']['percentage']:.2f}%)"
)


print()
print("Peatland disturbance drivers")
print("-" * 75)


print(
    f"Canal proximity  : "
    f"{r['drivers']['canal_proximity']}"
)


if (
    r["drivers"]["canal_distance_m"]
    is not None
):

    print(
        f"Nearest canal    : "
        f"{r['drivers']['canal_distance_m']:,.2f} m"
    )


print(
    f"Drainage pressure: "
    f"{r['drivers']['drainage_pressure']}"
)


print(
    f"Fire risk        : "
    f"{r['drivers']['fire_risk']}"
)

---
## Save

Each section above already ran and displayed itself. This cell writes them to one combined JSON.

In [ ]:
path = save_results(results, aoi, aoi_id, STAGE_GENERAL)
print(f"Saved {path}")